# AlphaZero API

- Reuse the shared Tic-Tac-Toe rules
- Explore board encodings, action masks, and exact outcomes
- Inspect policy normalization and the uniform policy/value evaluator
- Trace PUCT search, alternating value backups, and visit-count policies
- Implementation lives in `alphazero_utils.py`
- See [README.md](README.md) for setup and API conventions
- Inspect a CPU policy/value network and fit hand-built training targets

## Imports and Setup

- Launch from the repository environment documented in [README.md](README.md)
- The repository and `helpers_root` must be on `PYTHONPATH`

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import trange
from IPython.display import display

import helpers.hdbg as hdbg
import research.Implement_AlphaZero.alphazero_utils as rialzut
import research.Implement_MonteCarlo_Tree_Search_and_Alpha_Zero.game_examples as rimtsaazge

hdbg.init_logger(verbosity=logging.INFO)
_LOG = logging.getLogger(__name__)

# Part 1: States and Actions

| API | Result | Purpose |
| :--- | :--- | :--- |
| `encode_state(game, state)` | Flat `float32` vector | Model input from the current player's perspective |
| `get_legal_action_mask(game, state, action_size)` | Boolean vector | Legal entries in a fixed policy output |
| `get_terminal_value(game, state)` | Float or `None` | Exact terminal outcome, distinct from unfinished play |

## Cell 1.1: Reuse the Game

- The original state uses `1` for X, `-1` for O, and `0` for empty
- Actions are row-major indices: top row `0, 1, 2`, middle `3, 4, 5`,
  bottom `6, 7, 8`

In [ ]:
# Print is intentional throughout: this notebook exposes the API results.
game = rimtsaazge.TicTacToe()
state = game.get_initial_state()
print("state=", state)
print("board=\n" + game.render(state))
print("legal_moves=", game.get_legal_moves(state))

## Cell 1.2: Encode the Empty Board

- The model input is a flat vector of nine numbers
- An encoding is a fresh array; the tuple state remains owned by the game

In [ ]:
# Inspect the model input shape and dtype.
encoded = rialzut.encode_state(game, state)
print("encoded=", encoded)
print("shape=", encoded.shape, "dtype=", encoded.dtype)
np.testing.assert_array_equal(encoded, [0] * 9)

## Cell 1.3: Change the Player's Perspective

- X plays the center, then O is the player to move
- O sees X's center piece as `-1`; its own pieces are encoded as `+1`
- Multiply cell signs by the current player; keep cell positions unchanged
- Try changing `move` from `4` to another index and rerun this cell

In [ ]:
# Always start this experiment from an empty board so it can be rerun.
move = 4
state = game.apply_move(game.get_initial_state(), move)
encoded = rialzut.encode_state(game, state)
print("current_player=", game.get_current_player(state))
print("board=\n" + game.render(state))
print("encoded=", encoded)
hdbg.dassert_eq(encoded[move], -1.0, "O sees X's piece as an opponent")

- Never feed `encoded` into `apply_move()` or other game-rule methods
- Rules use the original `state`; encoding supplies the model input

## Cell 1.4: Preserve a Fixed Action Space

- All nine policy entries retain their original indices
- The occupied cell is False, but the mask still has length nine
- A mask indicates legality; a policy distribution assigns probabilities

In [ ]:
# Compare the fixed-size mask with the game's shorter list of legal moves.
mask = rialzut.get_legal_action_mask(game, state, 9)
print("legal_action_mask=", mask)
print("legal_indices=", np.flatnonzero(mask))
np.testing.assert_array_equal(np.flatnonzero(mask), game.get_legal_moves(state))

# Part 2: Exact Outcomes

## Cell 2.1: An Unfinished Game Has No Exact Value Yet

- `None` means unfinished; it does not mean a draw
- Nonterminal positions require value estimates rather than exact outcomes

In [ ]:
# The one-move board has no terminal outcome.
value = rialzut.get_terminal_value(game, state)
print("terminal_value=", value)
hdbg.dassert_is(value, None, "An unfinished game has no exact outcome")

## Cell 2.2: Walk Through a Complete Game

- Use the specified moves `0, 3, 1, 4, 2`; X wins across the top row
- Check every action against the mask before applying it
- These are hand-chosen moves, not an agent or self-play data collector

In [ ]:
# Trace the public game and representation APIs.
state = game.get_initial_state()
for move in [0, 3, 1, 4, 2]:
    mask = rialzut.get_legal_action_mask(game, state, 9)
    hdbg.dassert(mask[move], "The demonstration must use a legal move")
    state = game.apply_move(state, move)
    print(
        "move=",
        move,
        "terminal_value=",
        rialzut.get_terminal_value(game, state),
    )
print("final_board=\n" + game.render(state))

## Cell 2.3: Interpret a Win From the Next Player's Perspective

- Winner: X (`1`); next player under the game convention: O (`-1`)
- Value: winner times next player, hence `-1.0`
- Negate this value to express the outcome from X's perspective
- The game is over: empty cells must also be masked
- The existing MCTS stores values for the player who entered a node;
  its convention differs from this player-to-move value

In [ ]:
# Confirm the exact outcome and legal-action mask at a terminal state.
value = rialzut.get_terminal_value(game, state)
mask = rialzut.get_legal_action_mask(game, state, 9)
print("winner=", game.get_winner(state))
print("next_player=", game.get_current_player(state))
print("terminal_value=", value)
print("legal_action_mask=", mask)
hdbg.dassert_eq(value, -1.0, "O has lost after X wins")
np.testing.assert_array_equal(mask, [False] * 9)

## Cell 2.4: Distinguish a Draw

- This full board has no winning line
- A completed draw returns `0.0`, unlike the unfinished game's `None`

In [ ]:
# Inspect a reachable drawn position.
draw_state = (1, -1, 1, 1, -1, -1, -1, 1, 1)
value = rialzut.get_terminal_value(game, draw_state)
print("draw_board=\n" + game.render(draw_state))
print("terminal_value=", value)
hdbg.dassert_eq(value, 0.0, "A draw has zero value")

# Part 3: Legal Action Priors

| API | Purpose |
| :--- | :--- |
| `PolicyValuePrediction(policy, value)` | Validate and hold the evaluator's result |
| `PolicyValueEvaluator` | Callable signature: `(game, state) -> PolicyValuePrediction` |
| `normalize_policy(weights, legal_action_mask)` | Turn nonnegative weights into legal probabilities |
| `UniformEvaluator(action_size)` | Equal legal priors and a neutral nonterminal estimate |

## Cell 3.1: Evaluate an Empty Board

- Actions keep their row-major indices `0` through `8`
- Each action on an empty board receives probability `1/9`
- `value=0.0` is a neutral estimate, not a claim that play must end in a draw
- The evaluator neither chooses a move nor performs search or learning

In [ ]:
# Print is intentional: expose the numerical evaluator contract.
game = rimtsaazge.TicTacToe()
evaluator: rialzut.PolicyValueEvaluator = rialzut.UniformEvaluator(9)
state = game.get_initial_state()
prediction = evaluator(game, state)
print("policy=", prediction.policy)
print("value=", prediction.value)
np.testing.assert_allclose(prediction.policy, np.full(9, 1 / 9))
hdbg.dassert_eq(prediction.value, 0.0, "The baseline estimate is neutral")

## Cell 3.2: Mask an Occupied Cell

- X plays the center; O is now the player to move
- The center receives zero probability and each remaining action receives `1/8`
- The output always contains nine entries; legal moves are not renumbered
- Repeated evaluation returns the same result without sampling

In [ ]:
# Evaluate the original state after X occupies the center.
state = game.apply_move(game.get_initial_state(), 4)
mask = rialzut.get_legal_action_mask(game, state, 9)
prediction = evaluator(game, state)
print("board=\n" + game.render(state))
print("player_to_move=", game.get_current_player(state))
print("legal_action_mask=", mask)
print("policy=", prediction.policy)
np.testing.assert_allclose(prediction.policy, mask.astype(float) / 8)
np.testing.assert_array_equal(evaluator(game, state).policy, prediction.policy)

## Cell 3.3: Normalize Manually Chosen Weights

- Assign twice as much weight to each corner as to each edge
- Give the occupied center a large weight; it must still receive probability zero
- The normalizer accepts finite nonnegative weights, not arbitrary logits
- Each legal corner receives `2/12` and each edge receives `1/12`
- Try changing the weights and rerunning this cell to see the priors change

In [ ]:
# Mask first, then normalize only the legal weights.
weights = np.array([2, 1, 2, 1, 100, 1, 2, 1, 2], dtype=float)
policy = rialzut.normalize_policy(weights, mask)
print("weights=", weights)
print("normalized_policy=", policy)
print("policy_sum=", policy.sum())
np.testing.assert_allclose(policy, np.array([2, 1, 2, 1, 0, 1, 2, 1, 2]) / 12)

## Cell 3.4: Handle Zero Legal Weight

- Positive weight only on an illegal action leaves zero legal mass
- Fall back to equal probabilities over legal actions
- An all-False mask instead returns all zeros, representing no available policy
- Negative or nonfinite weights and mismatched masks are rejected

In [ ]:
# Removing the sole weighted action triggers the uniform fallback.
zero_legal_weights = np.zeros(9)
zero_legal_weights[4] = 100
fallback = rialzut.normalize_policy(zero_legal_weights, mask)
print("fallback_policy=", fallback)
np.testing.assert_allclose(fallback, prediction.policy)

# Part 4: Values and Terminal States

## Cell 4.1: Construct a Policy/Value Prediction

- A prediction has a normalized policy and a finite scalar value in `[-1, 1]`
- Positive values favor the player to move; negative values favor the opponent
- This manually chosen value illustrates the contract; it is not a model result
- The constructor validates numerical constraints and copies the policy
- The evaluator is responsible for game-specific legality and value perspective

In [ ]:
# Pair the hand-chosen legal prior with an illustrative value estimate.
manual_prediction = rialzut.PolicyValuePrediction(policy, 0.25)
print("manual_policy=", manual_prediction.policy)
print("manual_value=", manual_prediction.value)
print("uniform_value=", prediction.value)
hdbg.dassert_eq(manual_prediction.value, 0.25, "Preserve the supplied estimate")

## Cell 4.2: Use an Exact Outcome at a Win

- X has won across the top row; the game's next-player convention reports O
- O's exact value is `-1.0`, overriding the baseline's neutral estimate
- All policy entries are zero, including the still-empty cells
- A zero terminal policy is a sentinel, not a distribution to sample from
- Negating this value expresses the result from X's perspective

In [ ]:
# Terminal values come from the game rules.
won_state = (1, 1, 1, -1, -1, 0, 0, 0, 0)
terminal_prediction = evaluator(game, won_state)
print("board=\n" + game.render(won_state))
print("winner=", game.get_winner(won_state))
print("player_to_move=", game.get_current_player(won_state))
print("policy=", terminal_prediction.policy)
print("value=", terminal_prediction.value)
np.testing.assert_array_equal(terminal_prediction.policy, np.zeros(9))
hdbg.dassert_eq(terminal_prediction.value, -1.0, "O has lost")

## Cell 4.3: Distinguish a Draw From a Neutral Estimate

- A completed draw has exact value `0.0` and an all-zero policy
- An unfinished board has legal action probabilities and an estimated value
- Use the game's terminal check when deciding whether to act

In [ ]:
# Compare a completed draw with the unfinished center-opening position.
draw_state = (1, -1, 1, 1, -1, -1, -1, 1, 1)
draw_prediction = evaluator(game, draw_state)
print("draw_board=\n" + game.render(draw_state))
print("draw_value=", draw_prediction.value)
print("draw_policy=", draw_prediction.policy)
print("unfinished_exact_value=", rialzut.get_terminal_value(game, state))
print("unfinished_estimated_value=", evaluator(game, state).value)
np.testing.assert_array_equal(draw_prediction.policy, np.zeros(9))
hdbg.dassert_eq(draw_prediction.value, 0.0, "A completed draw has exact value zero")

# Part 5: PUCT Search Mechanics

- Selection combines an estimated action value with prior-weighted exploration
- Each node stores its value for the player to move in that node
- The parent uses `-child.mean_value`, because its player is the opponent
- A simulation evaluates one leaf and backs up its value with alternating signs
- Terminal leaves use the exact game outcome; other leaves use the evaluator

| API | Purpose |
| :--- | :--- |
| `AlphaZeroNode(state, prior)` | State, incoming prior, children, visits, and value sum |
| `get_puct_scores(node, exploration_constant)` | Inspect the scores used during selection |
| `build_search_tree(game, state, evaluator, ...)` | Build a fresh tree with an explicit simulation budget |
| `get_visit_policy(root, action_size)` | Normalize visits over the original action indices |

## Cell 5.1: Compute a Selection Score by Hand

- Consider one candidate child in a hand-constructed statistics snapshot
- The parent has 16 visits; the child has prior 0.25, three visits, and value sum -1.5
- The child's mean is -0.5; the parent sees the action value as +0.5
- With exploration constant 2, the bonus is `2 * 0.25 * sqrt(16) / 4 = 0.5`
- The total score is therefore 1.0

$$
\operatorname{score}(s,a) = -\overline{v}(s') + c P(s,a)
\frac{\sqrt{\max(1,N(s))}}{1+N(s,a)}
$$

- At an unvisited parent, `max(1, N)` makes the first choice honor the priors
- Unvisited children have mean zero; equal scores choose the lowest action index
- The turn-sign rule assumes strictly alternating two-player games

In [ ]:
# Build only the candidate needed to inspect this score.
game = rimtsaazge.TicTacToe()
score_root = rialzut.AlphaZeroNode(game.get_initial_state(), 1.0)
score_root.visit_count = 16
score_child = rialzut.AlphaZeroNode(game.apply_move(score_root.state, 0), 0.25)
score_child.visit_count = 3
score_child.value_sum = -1.5
score_root.children[0] = score_child
score = rialzut.get_puct_scores(score_root, 2.0)[0]
print("child_mean=", score_child.mean_value)
print("parent_action_value=", -score_child.mean_value)
print("puct_score=", score)
hdbg.dassert_eq(score, 1.0, "The score must match the hand calculation")

## Cell 5.2: Expand the Root Without Simulations

- X can win at action 2; O also threatens action 5
- Root expansion evaluates the current position and creates all legal children
- Expansion happens before the counted simulations; it does not back up the root estimate
- With a zero budget, visits are all zero and `get_visit_policy()` returns the priors
- The uniform evaluator does not identify the win; search must reach the terminal state

In [ ]:
# Inspect the legal priors before any simulation can find the winning move.
search_state = (1, 1, 0, -1, -1, 0, 0, 0, 0)
evaluator = rialzut.UniformEvaluator(9)
prior_root = rialzut.build_search_tree(
    game, search_state, evaluator, action_size=9, num_simulations=0
)
print("board=\n" + game.render(search_state))
print("root_visits=", prior_root.visit_count)
print("prior_policy=", rialzut.get_visit_policy(prior_root, 9))
print("initial_scores=", rialzut.get_puct_scores(prior_root, 1.0))
hdbg.dassert_eq(prior_root.visit_count, 0, "Expansion is outside the simulation budget")

## Cell 5.3: Trace One Simulation and Its Backup

- Equal priors select the lowest legal action, 2
- That move wins for X; the terminal child reports O as next and has value -1
- The evaluator is bypassed at this leaf; no random rollout is needed
- Backup records -1 at the child, then negates it to +1 at the root
- Each node on the path receives exactly one visit

```text
X to move: root              N=1, W=+1
  action 2 -> O to move      N=1, W=-1  (terminal)
```

- Across a two-level path, the signs flip twice: leaf -1, parent +1, grandparent -1
- Values from repeated simulations are averaged; the backup is not a maximum

In [ ]:
# Trace the first actual simulation from the same position.
one_root = rialzut.build_search_tree(
    game, search_state, evaluator, action_size=9, num_simulations=1
)
winning_child = one_root.children[2]
print("child_board=\n" + game.render(winning_child.state))
print("child_visits=", winning_child.visit_count, "child_value_sum=", winning_child.value_sum)
print("root_visits=", one_root.visit_count, "root_value_sum=", one_root.value_sum)
hdbg.dassert_eq(winning_child.value_sum, -1.0, "O loses at the leaf")
hdbg.dassert_eq(one_root.value_sum, 1.0, "X wins at the parent")

# Part 6: Search Decisions

## Cell 6.1: Turn Visits Into an Action Policy

- Run a larger budget from the same position and inspect the root's children
- Child means use the opponent's perspective; parent action values negate them
- Normalize the root child counts to obtain a policy; occupied cells retain probability zero
- Choose the most-visited action using `argmax`; equal counts choose the lowest index
- Root visits and the sum of root child visits both equal the requested budget
- Try changing `num_simulations` or `exploration_constant` and rerun the cell

In [ ]:
# Display search statistics alongside the resulting policy.
num_simulations = 100
root = rialzut.build_search_tree(
    game, search_state, evaluator, action_size=9,
    num_simulations=num_simulations, exploration_constant=1.0
)
visit_policy = rialzut.get_visit_policy(root, 9)
stats = pd.DataFrame([
    {"action": action, "prior": child.prior, "visits": child.visit_count,
     "child_mean": child.mean_value, "parent_action_value": -child.mean_value,
     "visit_probability": visit_policy[action]}
    for action, child in root.children.items()
])
display(stats)
print("visit_policy=", visit_policy)
print("selected_move=", int(np.argmax(visit_policy)))
hdbg.dassert_eq(root.visit_count, num_simulations, "Each simulation visits the root")
hdbg.dassert_eq(sum(c.visit_count for c in root.children.values()), num_simulations,
               "Each simulation selects one root child")

## Cell 6.2: Block an Opponent's Immediate Win

- O is to move; X threatens the top row at action 2
- The search must examine X's reply to recognize why other actions lose
- This example exercises alternating value signs beyond an immediate win
- Neutral leaf estimates and a finite budget do not guarantee optimal play in every position

In [ ]:
# Search from O's perspective and verify the forced block.
block_state = (1, 1, 0, 0, -1, 0, 0, 0, 0)
block_root = rialzut.build_search_tree(
    game, block_state, evaluator, action_size=9, num_simulations=200
)
block_policy = rialzut.get_visit_policy(block_root, 9)
block_move = int(np.argmax(block_policy))
print("board=\n" + game.render(block_state))
print("player_to_move=", game.get_current_player(block_state))
print("visit_policy=", block_policy)
print("selected_move=", block_move)
hdbg.dassert_eq(block_move, 2, "O must block X's top-row threat")

## Cell 6.3: Play a Complete Game With Search

- Both players use the same uniform evaluator and rebuild the tree at each move
- Each decision uses 64 simulations, no root noise, and deterministic tie-breaking
- This is a search demonstration; no training examples are collected and no parameters are updated
- No conclusion about playing strength follows from a single game

In [ ]:
# Apply each search decision to the original immutable game state.
play_state = game.get_initial_state()
moves = []
while not game.is_terminal(play_state):
    play_root = rialzut.build_search_tree(
        game, play_state, evaluator, action_size=9, num_simulations=64
    )
    move = int(np.argmax(rialzut.get_visit_policy(play_root, 9)))
    hdbg.dassert_in(move, game.get_legal_moves(play_state), "Search must choose a legal action")
    moves.append(move)
    play_state = game.apply_move(play_state, move)
print("moves=", moves)
print("final_board=\n" + game.render(play_state))
print("winner=", game.get_winner(play_state))
hdbg.dassert(game.is_terminal(play_state), "The game must finish")

# Part 7: A Policy/Value Network

- A shared multilayer perceptron (MLP) reads the current-player encoding
- Two ReLU hidden layers feed separate policy and value heads
- The policy head returns raw logits for every action; the value head uses
  $\tanh$ to bound its estimate to $[-1, 1]$
- The model uses CPU float32 tensors; board size and action count are independent

| API | Purpose | Input/output convention |
| :--- | :--- | :--- |
| `PolicyValueNetwork(input_size, action_size, hidden_size=..., seed=...)` | Construct the MLP | Explicit local seed; CPU float32 parameters |
| `network(encoded_tensor)` | Compute differentiable predictions | One board or a batch; logits and player-to-move values |
| `NetworkEvaluator(network)` | Supply predictions to PUCT | Original game state in; legal probabilities and scalar value out |
| `train_batch(network, optimizer, inputs, policies, values, l2_coefficient=...)` | Take one supervised optimizer step | NumPy batches in; pre-update loss components out |

## Cell 7.1: Inspect Single and Batched Predictions

- Tic-Tac-Toe has nine input cells and nine policy logits
- A single board yields logits shaped `(9,)` and a scalar value
- A batch of two boards yields `(2, 9)` logits and `(2,)` values
- The initialization seed is local to construction and preserves the caller's CPU RNG

In [ ]:
# Small matrix operations benefit from a single CPU thread in this notebook.
torch.set_num_threads(1)
game = rimtsaazge.TicTacToe()
network = rialzut.PolicyValueNetwork(9, 9, hidden_size=32, seed=7)
network_evaluator = rialzut.NetworkEvaluator(network)
example_state = game.apply_move(game.get_initial_state(), 4)
example_input = torch.from_numpy(rialzut.encode_state(game, example_state))
with torch.no_grad():
    single_logits, single_value = network(example_input)
    batch_logits, batch_values = network(torch.stack([example_input, example_input]))
display(pd.DataFrame([
    {"input": "one board", "logits_shape": tuple(single_logits.shape), "value_shape": tuple(single_value.shape)},
    {"input": "two boards", "logits_shape": tuple(batch_logits.shape), "value_shape": tuple(batch_values.shape)},
]))
print("network=", network)
hdbg.dassert_eq(tuple(batch_logits.shape), (2, 9), "One logit per action per board")
hdbg.dassert((batch_values.abs() <= 1).all().item(), "Values must be bounded")

## Cell 7.2: Convert Logits Into Legal Priors

- Raw logits can be negative and do not sum to one
- The evaluator masks illegal logits before softmax, then returns probabilities
- The occupied center must have zero probability even if its raw logit is large
- Untrained values are estimates, without evidence of playing strength

In [ ]:
# Compare unrestricted model probabilities with game-aware evaluation.
unmasked_policy = torch.softmax(single_logits, dim=-1).numpy()
network_prediction = network_evaluator(game, example_state)
display(pd.DataFrame({
    "action": np.arange(9),
    "legal": rialzut.get_legal_action_mask(game, example_state, 9),
    "logit": single_logits.numpy(),
    "raw_probability": unmasked_policy,
    "legal_prior": network_prediction.policy,
}))
print("predicted_value=", network_prediction.value)
hdbg.dassert_eq(network_prediction.policy[4], 0.0, "Occupied center must be masked")
np.testing.assert_allclose(network_prediction.policy.sum(), 1.0)
# Finished games still use exact rule-based outcomes, bypassing the model.
terminal = network_evaluator(game, (1, 1, 1, -1, -1, 0, 0, 0, 0))
hdbg.dassert_eq(terminal.value, -1.0, "The next player has lost")
np.testing.assert_array_equal(terminal.policy, np.zeros(9))

# Part 8: Learning From Controlled Targets

- Train on four reachable, hand-built positions to inspect learning in isolation
- Each policy target is a chosen legal move encoded as a distribution
- Values label the player to move: immediate wins $+1$, a forced draw $0$,
  and a forced loss $-1$
- These examples demonstrate fitting; they do not measure generalization or
  establish that the network plays well

## Cell 8.1: Build the Fixed Dataset

- X wins at cell 2 in the first board; O wins at cell 5 in the second
- X fills the last cell, 8, for a draw in the third
- O loses the fourth board: X threatens cells 2 and 6 simultaneously;
  the policy target chooses the legal block at 2, but its value remains $-1$

In [ ]:
# Keep action coordinates fixed while encoding each board for its own player.
training_states = [
    (1, 1, 0, -1, -1, 0, 0, 0, 0),
    (1, 1, 0, -1, -1, 0, 1, 0, 0),
    (1, -1, 1, 1, -1, -1, -1, 1, 0),
    (1, 1, 0, 1, -1, 0, 0, 0, 0),
]
target_actions = np.array([2, 5, 8, 2])
training_inputs = np.stack([rialzut.encode_state(game, state) for state in training_states])
target_policies = np.eye(9, dtype=np.float32)[target_actions]
target_values = np.array([1, 1, 0, -1], dtype=np.float32)
display(pd.DataFrame({
    "state": training_states,
    "player_to_move": [game.get_current_player(state) for state in training_states],
    "target_action": target_actions,
    "target_value": target_values,
}))
for state, action in zip(training_states, target_actions):
    hdbg.dassert_in(int(action), game.get_legal_moves(state), "Targets must be legal")

## Cell 8.2: Record Predictions Before Training

- Reset the seed and network here so rerunning the experiment is reproducible
- Retain both raw policy probabilities and game-aware evaluator predictions
- The evaluator holds the model by reference and will see later parameter updates

In [ ]:
network = rialzut.PolicyValueNetwork(9, 9, hidden_size=32, seed=7)
network_evaluator = rialzut.NetworkEvaluator(network)
with torch.no_grad():
    before_logits, before_values = network(torch.from_numpy(training_inputs))
    before_probabilities = torch.softmax(before_logits, dim=-1).numpy()
before_legal = [network_evaluator(game, state) for state in training_states]
display(pd.DataFrame({
    "target_action": target_actions,
    "raw_target_probability": before_probabilities[np.arange(4), target_actions],
    "legal_target_probability": [prediction.policy[action] for prediction, action in zip(before_legal, target_actions)],
    "predicted_value": before_values.numpy(),
    "target_value": target_values,
}))

## Cell 8.3: Fit Both Heads With One Objective

For batch size $B$, use

$$
L = -\frac{1}{B}\sum_{i,a}\pi_{i,a}\log\operatorname{softmax}(\ell_i)_a
    + \frac{1}{B}\sum_i(v_i-z_i)^2
    + c\sum_{\theta_j}\theta_j^2.
$$

- Cross-entropy trains raw logits over **all** actions; illegal actions have
  zero target mass but remain in the softmax denominator
- The implementation accepts soft policy targets as well as the one-hot examples
- Mean squared error trains values from the player-to-move perspective
- L2 includes every parameter, including biases; keep optimizer weight decay
  zero to avoid adding regularization twice
- Reuse one Adam optimizer across updates to retain its moment estimates
- Returned losses describe each batch **before** its optimizer update

In [ ]:
# Fit the same four examples; no game generation is involved in this experiment.
optimizer = torch.optim.Adam(network.parameters(), lr=0.02, weight_decay=0.0)
training_history = []
for step in trange(150, desc="Fit fixed targets"):
    metrics = rialzut.train_batch(
        network, optimizer, training_inputs, target_policies, target_values,
        l2_coefficient=1e-4,
    )
    training_history.append({"step": step + 1, **metrics})
loss_history = pd.DataFrame(training_history).set_index("step")
display(loss_history.iloc[[0, 24, 74, 149]])
np.testing.assert_array_less(loss_history["loss"].iloc[-1], loss_history["loss"].iloc[0] * 0.1)

## Cell 8.4: Inspect the Loss Components

- Policy and value errors should fall on this fixed dataset
- The weighted L2 term can rise as the model uses larger weights to fit the targets
- These are training losses; there is no held-out evaluation in this experiment

In [ ]:
# Plot the observed objective components using the recorded pre-update metrics.
axis = loss_history[["loss", "policy_loss", "value_loss", "l2_loss"]].plot(
    figsize=(8, 4), logy=True, title="Fitting four controlled board positions"
)
axis.set_xlabel("Optimizer step")
axis.set_ylabel("Pre-update loss (log scale)")
axis.grid(alpha=0.25)
# Explicit display also renders the figure in headless notebook runs.
display(axis.figure)
plt.close(axis.figure)

## Cell 8.5: Compare Predictions After Fitting

- Compare raw target probabilities to avoid hiding mistakes through legal masking
- The value head should separate wins, draws, and losses
- The same evaluator instance now uses the updated network

In [ ]:
with torch.no_grad():
    after_logits, after_values = network(torch.from_numpy(training_inputs))
    after_probabilities = torch.softmax(after_logits, dim=-1).numpy()
after_legal = [network_evaluator(game, state) for state in training_states]
display(pd.DataFrame({
    "target_action": target_actions,
    "probability_before": before_probabilities[np.arange(4), target_actions],
    "probability_after": after_probabilities[np.arange(4), target_actions],
    "value_before": before_values.numpy(),
    "value_after": after_values.numpy(),
    "target_value": target_values,
}))
hdbg.dassert((after_probabilities[np.arange(4), target_actions] > 0.95).all(), "The fixed policy targets should fit")
np.testing.assert_allclose(after_values.numpy(), target_values, atol=0.1)
for state, prediction in zip(training_states, after_legal):
    legal = rialzut.get_legal_action_mask(game, state, 9)
    np.testing.assert_array_equal(prediction.policy[~legal], 0)
    np.testing.assert_allclose(prediction.policy.sum(), 1.0)

## Cell 8.6: Plug the Network Into PUCT

- The search API is unchanged: supply `NetworkEvaluator` as the evaluator
- Search still uses exact outcomes at terminal leaves
- The displayed position belongs to the training set; this is an integration
  example, not an independent test of playing strength

In [ ]:
network_root = rialzut.build_search_tree(
    game, training_states[0], network_evaluator,
    action_size=9, num_simulations=32,
)
network_visit_policy = rialzut.get_visit_policy(network_root, 9)
display(pd.DataFrame({
    "action": np.arange(9),
    "learned_legal_prior": after_legal[0].policy,
    "search_visit_policy": network_visit_policy,
}))
print("selected_action=", int(np.argmax(network_visit_policy)))
hdbg.dassert_eq(network_root.visit_count, 32, "Network evaluation preserves the search budget")
hdbg.dassert_eq(int(np.argmax(network_visit_policy)), 2, "Choose the demonstrated immediate win")

## Summary: The Mental Model

- Original game states determine legality and exact outcomes; player-relative
  encodings supply network inputs
- The network returns logits and a bounded value; the evaluator converts them
  into legal priors and a player-to-move estimate for PUCT
- One training step combines policy cross-entropy, value regression, and explicit
  L2 regularization while retaining optimizer state between calls
- Fitting four supervised examples verifies learning mechanics; broader playing
  strength requires separate evidence